# Type Hints in Python

Type hints let you declare the **expected data type** of variables, function parameters, and return values — right in the source code.

**Critical rule:** Python **NEVER enforces** type hints at runtime. They are purely informational.

Despite not being enforced, type hints have become increasingly important in modern Python projects. Here's why:

| Benefit | Explanation |
|---|---|
| **IDE intelligence** | Editors (VS Code, PyCharm) use hints to power autocomplete, refactoring, and inline docs |
| **Static analysis** | Tools like `mypy`, `pyright` catch type errors *before* running the code |
| **Readability** | Type hints serve as executable documentation — tells readers what a function expects |
| **Maintenance** | Large codebases become much easier to navigate and refactor safely |
| **Onboarding** | New developers understand APIs instantly without reading implementation |

The `typing` module (introduced in Python 3.5) provides the building blocks for expressive type annotations.

---

# Type Hints for Standard Python Objects

The most basic form of type annotation uses Python's built-in type names directly.

**Syntax:** `variable_name: type = value`

This is called a **variable annotation**. The colon introduces the type, and the `= value` part is optional.

In [ ]:
# Basic variable annotations using built-in Python types
# Syntax: variable_name: type = value

# Annotate emp_name as a str — it should hold a string value
emp_name: str = 'Jim'

# Annotate emp_year as an int — it should hold a whole number
emp_year: int = 2021

# Annotate emp_task as a list — it should hold a list (of anything)
emp_task: list = []

# Annotate emp_proj as a tuple — it should hold a tuple (of anything)
emp_proj: tuple = ()

# Annotate emp_info as a dict — it should hold a dictionary (any keys/values)
emp_info: dict = {}

# These annotations work and are readable, but they're imprecise for containers.
# list/tuple/dict just says 'it's a container' — but WHAT does it contain?
# For that, use the typing module (shown next).

## Richer Annotations with the `typing` Module

For containers (`list`, `tuple`, `dict`, `set`), the plain built-in type is not very informative. You know *it's a list*, but you don't know *what type of items* it holds.

The `typing` module provides generic versions that let you specify **element types** too.

> **Python 3.9+ shortcut:** You can use the lowercase `list[int]`, `tuple[str]`, `dict[int, str]` directly without importing from `typing`. The `typing.List`, `typing.Tuple`, `typing.Dict` forms work in older versions.

In [ ]:
# Import the generic container types from the typing module
# List, Tuple, Dict are parameterizable — they accept type arguments inside []
from typing import List, Tuple, Dict

# List[int]: a list whose elements are all integers
# This is much more informative than just 'list'
emp_task: List[int] = []        # e.g., [1, 2, 3] — task IDs as integers

# Tuple[str]: a tuple whose elements are strings
emp_proj: Tuple[str] = ()       # e.g., ('Alpha', 'Beta') — project names as strings

# Dict[int, str]: a dictionary with int keys and str values
# The first type argument is the key type, the second is the value type
emp_info: Dict[int, str] = {}   # e.g., {1: 'Alice', 2: 'Bob'} — emp_id → emp_name

# Now a reader immediately knows:
# - emp_task holds integers
# - emp_proj holds strings
# - emp_info maps integers to strings
print("Type-annotated containers defined successfully")

---

# `Union` — Multiple Allowed Types

Sometimes a variable can legitimately hold **more than one type** of value. For example, a mathematical result might be `int` or `float` depending on the input.

`Union[X, Y]` means: *"This value can be of type X **or** type Y."*

> **Python 3.10+ shortcut:** You can use the `|` (pipe) operator instead of `Union`. Both are equivalent.

In [ ]:
# Import Union to declare variables that can hold one of several types
from typing import Union

side = 3

# Union[int, float]: the area variable can be either int OR float
# Why? If side=3 (int), area = 9 (int). If side=3.5 (float), area = 12.25 (float)
# Union captures this ambiguity honestly
area: Union[int, float] = side * side

print(f"area = {area}, type = {type(area).__name__}")

In [ ]:
# Python 3.10+ introduced the | (pipe) operator as shorthand for Union
# int|float is identical in meaning to Union[int, float]
# This is cleaner and more readable for simple cases
area: int | float = side * side  # Requires Python 3.10+

print(f"area = {area}, type = {type(area).__name__}")

# More Union examples:
# user_id: Union[int, str] = 42        # Could be numeric or string ID
# result: Union[int, float, None] = None  # Success value or None on failure

---

# `Optional` — A Value That Might Be `None`

`Optional[X]` is a very common shorthand for `Union[X, None]`.

It expresses: *"This value is normally of type X, but it might also be `None`.*"

This is especially useful for:
- Function parameters with default `None` values
- Return types that indicate "not found" with `None`
- Attributes that haven't been set yet

In [ ]:
# Import Optional — shorthand for Union[X, None]
from typing import Optional

# Optional[int] means this can be an int OR None
# Equivalent to: count: Union[int, None] = 10
count: Optional[int] = 10  # Currently an int, but may be set to None later

print(f"count = {count}")

# Setting to None is perfectly valid — that's exactly what Optional allows
count = None
print(f"count after reset = {count}")

# Real-world example: a function that finds a user by ID
# Returns None if the user doesn't exist
def find_user(user_id: int) -> Optional[str]:
    users = {1: 'Alice', 2: 'Bob'}
    # .get() returns None if key not found — matches our Optional[str] return type
    return users.get(user_id)

print(find_user(1))   # → 'Alice'
print(find_user(99))  # → None

---

# `Iterable` — Anything You Can Loop Over

An **iterable** is any object Python can iterate over with a `for` loop: `list`, `tuple`, `set`, `dict`, `generator`, `range`, etc.

`Iterable` as a type hint says: *"I accept any object you can loop over — I don't care which specific container type it is."*

This is the **most flexible** container hint and follows Python's duck-typing philosophy.

In [ ]:
# Import Iterable — accepts any object that supports iteration
from typing import Iterable

# arr: Iterable means this function accepts any iterable — list, tuple, set, dict, etc.
# Using 'Iterable' instead of 'list' makes the function more general and reusable
def total_sum(arr: Iterable):
    # sum() works on ANY iterable, so using Iterable as the hint is honest and flexible
    print(f'Type(arr):{type(arr)}, arr:{arr}, sum(arr): {sum(arr)}')

# ✅ Passing a list — Iterable is satisfied
total_sum([1, 2, 3, 4, 5])

# ✅ Passing a set — also an Iterable
total_sum({1, 2, 3, 4, 5})

# ✅ Passing a tuple — also an Iterable
total_sum((1, 2, 3, 4, 5))

# ✅ Passing a dict — iterating over a dict yields its KEYS
# So sum() sums the keys (1+2+3+4+5=15), not the values
total_sum({1: 'A', 2: 'B', 3: 'C', 4: 'D', 5: 'E'})

---

# `Final` — Constant Values

`Final` marks a variable that **should never be reassigned** after its initial value is set.

It's Python's equivalent of `const` in JavaScript or `final` in Java.

Python does **not enforce** this at runtime, but static type checkers like `mypy` will flag any reassignment as an error.

In [ ]:
# Import Final — marks a variable as a constant that should never be reassigned
from typing import Final

# Final without a type argument lets Python infer the type (float here)
# Convention: constants are often written in UPPER_CASE
PI: Final = 3.1412  # Marks PI as a constant — mypy will error if you do: PI = 3.0

print(f"PI = {PI}")

# You can also specify the type explicitly:
# PI: Final[float] = 3.1412

# Other common uses of Final:
MAX_RETRIES: Final[int] = 3        # Configuration constant
API_BASE_URL: Final[str] = "https://api.example.com"  # Fixed endpoint

# Note: Python WON'T stop you from doing 'PI = 3.0' at runtime.
# Only static analysis tools (mypy, pyright) enforce Final.
print(f"MAX_RETRIES = {MAX_RETRIES}")

---

# `Literal` — Restricting to a Fixed Set of Values

`Literal` is different from `Final`. It doesn't say the variable is constant — it says the variable can only take **one of a specific set of allowed values**.

Think of it like an enum, but lighter weight.

`Literal['A', 'B', 'C']` means: *"This value must be exactly `'A'`, `'B'`, or `'C'` — no other string is allowed."*

In [ ]:
# Import Literal — restricts a variable to a specific set of literal values
from typing import Literal

# Declare that 'grade' may only ever hold one of these exact string values
# Assigning grade = 'Z' or grade = 'pass' would be flagged by mypy as a type error
grade: Literal['A', 'B', 'C', 'D', 'E', 'F']  # Declaration without assignment

# Assign a valid Literal value — 'A' is in the allowed set
grade = 'A'
print(f"grade = {grade}")

# Practical example: direction for movement
# Only exactly these four strings are valid — no typos like 'nort' allowed
Direction = Literal['north', 'south', 'east', 'west']

def move(direction: Direction) -> str:
    # A static type checker ensures direction must be one of the four literals
    return f'Moving {direction}'

print(move('north'))  # ✅ Valid
# print(move('up'))   # ❌ mypy would flag this — 'up' is not in the Literal set

---

# `Any` — Opting Out of Type Checking

`Any` is the escape hatch — it explicitly says *"this can be absolutely any type, and I don't want type checking applied here."*

It's compatible with every other type — a variable annotated as `Any` can be assigned any value and can be used anywhere.

> **Use sparingly.** Overusing `Any` defeats the purpose of type hints. It's useful during migration of untyped code, or for genuinely polymorphic data (e.g., JSON payloads).

In [ ]:
# Import Any — the 'wildcard' type that disables type checking for this variable
from typing import Any

# data: Any signals "I don't know or care what type this is"
# Type checkers will skip validation for 'data' — they trust you to handle it
data: Any = 10           # Currently an int
print(f"data = {data}, type = {type(data).__name__}")

data = "hello"           # Now it's a str — Any allows this
print(f"data = {data}, type = {type(data).__name__}")

data = [1, 2, 3]         # Now it's a list — still fine with Any
print(f"data = {data}, type = {type(data).__name__}")

# Common legitimate uses of Any:
# - Parsing raw JSON (structure unknown until runtime)
# - Legacy code being gradually typed
# - Highly generic utility functions

---

# Type Hints in Functions

Functions are where type hints are most valuable. You annotate:
1. **Parameters** — what the function expects as input
2. **Return type** — what the function produces as output (after the `->` arrow)

**Syntax:**
```python
def function_name(param1: Type1, param2: Type2) -> ReturnType:
    ...
```

All the types we've seen (`Union`, `Optional`, `Iterable`, `Any`, etc.) can be used in function signatures.

In [ ]:
# s1: str → parameter 's1' must be a string
# s2: str → parameter 's2' must be a string
# -> str  → this function returns a string
def munge_string(s1: str, s2: str) -> str:
    # Concatenate s1 and s2, then reverse the result with [::-1] slice
    # Example: 'hello' + 'world' = 'helloworld', reversed = 'dlrowolleh'
    return (s1 + s2)[::-1]

# Both arguments are strings → type hints are satisfied
result = munge_string("hello", "world")
print(result)  # → 'dlrowolleh'

In [ ]:
# Combining type hints in function signatures
from typing import List, Optional, Union

# A more complex example using multiple type hint constructs together
def compute_average(numbers: List[float], default: Optional[float] = None) -> Optional[float]:
    # numbers: List[float]       → expects a list of floats
    # default: Optional[float]   → optional parameter, can be float or None
    # -> Optional[float]         → returns a float, or None if list is empty and no default

    # Handle the edge case of an empty list
    if not numbers:
        # Return the default value (could be None or a float)
        return default

    # Calculate and return the arithmetic mean
    return sum(numbers) / len(numbers)

print(compute_average([1.0, 2.5, 3.0]))      # → 2.1666...
print(compute_average([], default=0.0))       # → 0.0
print(compute_average([]))                    # → None

---

# `Callable` — Type Hint for Function Objects

In Python, functions are first-class objects — they can be stored in variables, passed as arguments, and returned from other functions.

`Callable` is the type hint for *any object that can be called* (i.e., supports `object()` syntax).

This includes:
- Regular functions (`def` or `lambda`)
- Built-in functions (`len`, `sum`)
- Imported functions (`math.log10`)
- Class instances with `__call__` defined

In [ ]:
# Import Callable — the type for any object that can be invoked as a function
from typing import Callable

# func: Callable → accepts any callable (function, method, or callable object)
# n: int         → accepts an integer argument
# -> float       → returns a float
def foo(func: Callable, n: int) -> float:
    # Call whatever function was passed in, with n as the argument
    # The actual behaviour depends entirely on which callable is passed
    return func(n)

In [ ]:
# Example 1: Passing a user-defined function

# A polynomial function: f(n) = 2n² + 4n + 5
def bar(n):
    return 2 * n**2 + 4 * n + 5

# Pass 'bar' as the callable argument — it will be called as: bar(2)
result = foo(bar, 2)
print(f"foo(bar, 2) = {result}")  # → 2*(4) + 4*(2) + 5 = 8+8+5 = 21

In [ ]:
# Example 2: Passing an imported library function

import math

# math.log10 is a built-in function from the math module
# It's also Callable — so it satisfies the type hint
# This will compute: math.log10(100) = 2.0
result = foo(math.log10, 100)
print(f"foo(math.log10, 100) = {result}")  # → 2.0

In [ ]:
# Example 3: Passing an instance of a class with __call__ defined

class Munger:
    # Defining __call__ makes instances of Munger callable: munger("Hello")
    # This is called a 'functor' or 'callable object'
    def __call__(self, s: str) -> str:
        # Repeat the string 5 times, then reverse it
        # Example: 'Hi' → 'HiHiHiHiHi' → 'iHiHiHiHiH'
        return (s * 5)[::-1]

# Create an instance of Munger — it can be called like a function
munger = Munger()

# munger is a Callable — it satisfies the type hint perfectly
# This calls munger.__call__("Hello") under the hood
result = foo(munger, "Hello")
print(f"foo(munger, 'Hello') = {result}")

**More precise `Callable` signatures:**

You can optionally specify exact parameter types and return types for callables:

```python
from typing import Callable

# Callable[[int], float]:  a function that takes one int and returns a float
# Callable[[str, str], bool]:  a function that takes two strings and returns a bool
# Callable[..., Any]:  a function with any signature

def apply(func: Callable[[int], float], value: int) -> float:
    return func(value)
```

---

# `TypeAlias` — Naming Complex Type Hints

As your codebase grows, type hints can become long and repetitive. Repeating `Union[int, float, str]` everywhere is verbose and hard to maintain.

`TypeAlias` lets you give a **short, descriptive name** to a complex type hint — then reuse that name throughout your code.

Benefits:
- ✅ DRY principle — define the type once, use it everywhere
- ✅ Improved readability — `SalaryType` is clearer than `Union[int, float]`
- ✅ Single point of change — update the alias definition, all usages update automatically

In [ ]:
# Import TypeAlias (available since Python 3.10)
# This explicitly marks the assignment as a type alias (not a regular variable)
from typing import TypeAlias, Union

# Define a type alias: IntFloStr can be used wherever Union[int, float, str] would appear
# TypeAlias annotation signals to tools that this is a type definition, not a value
IntFloStr: TypeAlias = Union[int, float, str]

# Now use the alias instead of the full Union expression
data: IntFloStr = 10        # int — valid
print(f"data = {data}")

data = 3.14                 # float — valid
print(f"data = {data}")

data = "hello"              # str — valid
print(f"data = {data}")

In [ ]:
# Real-world example: type aliases for complex, reusable structures
from typing import TypeAlias, List, Dict, Optional, Tuple

# Instead of writing this verbose type everywhere...
# Dict[str, List[Tuple[int, str]]]

# ...define a readable alias:
# A record maps a string key to a list of (id, name) pairs
RecordEntry: TypeAlias = Tuple[int, str]          # e.g., (42, 'Alice')
RecordList: TypeAlias = List[RecordEntry]          # e.g., [(42, 'Alice'), (99, 'Bob')]
RecordMap: TypeAlias = Dict[str, RecordList]       # e.g., {'dept_A': [(42, 'Alice'), ...]}

# Now function signatures are readable and meaningful
def lookup_department(records: RecordMap, dept: str) -> Optional[RecordList]:
    # Returns the list of (id, name) pairs for a department, or None if not found
    return records.get(dept)

sample: RecordMap = {
    'Engineering': [(1, 'Alice'), (2, 'Bob')],
    'Marketing': [(3, 'Carol')]
}

print(lookup_department(sample, 'Engineering'))  # → [(1, 'Alice'), (2, 'Bob')]
print(lookup_department(sample, 'HR'))           # → None

---

## Summary — Type Hints Quick Reference

```python
from typing import List, Tuple, Dict, Union, Optional, Iterable, Final, Literal, Any, Callable, TypeAlias

# Basic types
name: str = 'Alice'
age: int = 30
score: float = 9.5
active: bool = True

# Containers
ids: List[int] = [1, 2, 3]
coords: Tuple[float, float] = (1.0, 2.0)
lookup: Dict[str, int] = {'a': 1}

# Multiple types
value: Union[int, str] = 42       # int OR str
value: int | str = 42             # Python 3.10+ shorthand

# Nullable (can be None)
result: Optional[str] = None      # str OR None

# Constants and restricted values
PI: Final[float] = 3.14159        # Cannot be reassigned
status: Literal['on', 'off']      # Only these exact values

# Any type
data: Any = ...                   # Opt out of type checking

# Flexible containers
def f(items: Iterable) -> None: ... # Accepts any iterable

# Callable objects
def apply(func: Callable, x: int): ... # func is any callable

# Type aliases
Numeric: TypeAlias = Union[int, float]  # Reusable alias
```

> **Remember:** Python never enforces these at runtime. Install `mypy` or `pyright` to get actual type-checking feedback.